In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
from mpl_toolkits.mplot3d import Axes3D

from google.colab import drive
drive.mount('/content/drive')


pkl_path = '/content/drive/MyDrive/HCI Project/Datasets/ibdts_processed_data/ibdts_processed_data/motion'  # Folder with .npy files
video_path = '/content/drive/MyDrive/HCI Project/Datasets/ibdts_processed_data/ibdts_processed_data/3D_Motion_Videos'  # Folder to save final videos
img_path = '/content/drive/MyDrive/HCI Project/Datasets/ibdts_processed_data/ibdts_processed_data/3D_Motion_Images'    # Temporary image frames

os.makedirs(video_path, exist_ok=True)
os.makedirs(img_path, exist_ok=True)


def load_pred_position(npy_path):
    """
    Robustly load 'pred_position' (or joint array) from a .npy file.
    Returns an ndarray shaped (T, 24, 3) or raises an informative error.
    """
    data = np.load(npy_path, allow_pickle=True)

    if isinstance(data, dict):
        if 'pred_position' in data:
            arr = data['pred_position']
        else:
            found = False
            for v in data.values():
                a = np.asarray(v)
                if a.ndim == 3 and a.shape[1] == 24 and a.shape[2] == 3:
                    arr = a
                    found = True
                    break
            if not found:
                raise ValueError(f"No 'pred_position' key and no (T,24,3) array found in dict: {npy_path}. Keys: {list(data.keys())}")
    elif isinstance(data, np.ndarray):
        if data.dtype == object and data.size == 1:
            inner = data.flatten()[0]
            if isinstance(inner, dict) and 'pred_position' in inner:
                arr = inner['pred_position']
            elif isinstance(inner, np.ndarray):
                arr = inner
            else:
                arr = np.asarray(inner)
        else:
            if data.ndim == 3 and data.shape[1] == 24 and data.shape[2] == 3:
                arr = data
            else:
                found = False
                for candidate in np.atleast_1d(data):
                    a = np.asarray(candidate)
                    if a.ndim == 3 and a.shape[1] == 24 and a.shape[2] == 3:
                        arr = a
                        found = True
                        break
                if not found:
                    if data.dtype == object:
                        for el in data.flatten():
                            if isinstance(el, dict) and 'pred_position' in el:
                                arr = el['pred_position']
                                found = True
                                break
                    if not found:
                        raise ValueError(f"Could not find (T,24,3) joint array or 'pred_position' in {npy_path}. Found ndarray shape {getattr(data, 'shape', None)} dtype {getattr(data, 'dtype', None)}")
    else:
        raise ValueError(f"Unrecognized np.load result type {type(data)} from {npy_path}")

    arr = np.asarray(arr)

    if arr.ndim == 3 and arr.shape[1] == 24 and arr.shape[2] == 3:
        return arr
    if arr.ndim == 4 and arr.shape[-2] == 24 and arr.shape[-1] == 3:
        new = arr.reshape(-1, arr.shape[-2], arr.shape[-1])

        return new[0]
    raise ValueError(f"Loaded array from {npy_path} does not have expected joint shape (T,24,3). Got shape {arr.shape}")


def adjust_pose(joint):
    np_dance_trans = np.zeros([3, 25]).copy()
    joint = np.transpose(joint)

    # head
    np_dance_trans[:, 0] = joint[:, 15]

    #neck
    np_dance_trans[:, 1] = joint[:, 12]

    # left up
    np_dance_trans[:, 2] = joint[:, 16]
    np_dance_trans[:, 3] = joint[:, 18]
    np_dance_trans[:, 4] = joint[:, 20]

    # right up
    np_dance_trans[:, 5] = joint[:, 17]
    np_dance_trans[:, 6] = joint[:, 19]
    np_dance_trans[:, 7] = joint[:, 21]

    np_dance_trans[:, 8] = joint[:, 0]

    np_dance_trans[:, 9] = joint[:, 1]
    np_dance_trans[:, 10] = joint[:, 4]
    np_dance_trans[:, 11] = joint[:, 7]

    np_dance_trans[:, 12] = joint[:, 2]
    np_dance_trans[:, 13] = joint[:, 5]
    np_dance_trans[:, 14] = joint[:, 8]

    np_dance_trans[:, 15] = joint[:, 15]
    np_dance_trans[:, 16] = joint[:, 15]
    np_dance_trans[:, 17] = joint[:, 15]
    np_dance_trans[:, 18] = joint[:, 15]

    np_dance_trans[:, 19] = joint[:, 11]
    np_dance_trans[:, 20] = joint[:, 11]
    np_dance_trans[:, 21] = joint[:, 8]

    np_dance_trans[:, 22] = joint[:, 10]
    np_dance_trans[:, 23] = joint[:, 10]
    np_dance_trans[:, 24] = joint[:, 7]

    np_dance_trans = np.transpose(np_dance_trans)

    return np_dance_trans


pose_edge_list = [
    [ 0,  1], [ 1,  8],                                         # body
    [ 1,  2], [ 2,  3], [ 3,  4],                               # right arm
    [ 1,  5], [ 5,  6], [ 6,  7],                               # left arm
    [ 8,  9], [ 9, 10], [10, 11], [11, 24], [11, 22], [22, 23], # right leg
    [ 8, 12], [12, 13], [13, 14], [14, 21], [14, 19], [19, 20]  # left leg
]
pose_color_list = [
    [153,  0, 51], [153,  0,  0],
    [153, 51,  0], [153,102,  0], [153,153,  0],
    [102,153,  0], [ 51,153,  0], [  0,153,  0],
    [  0,153, 51], [  0,153,102], [  0,153,153], [  0,153,153], [  0,153,153], [  0,153,153],
    [  0,102,153], [  0, 51,153], [  0,  0,153], [  0,  0,153], [  0,  0,153], [  0,  0,153]
]


def plot_line(joint, ax):
    for i, e in enumerate(pose_edge_list):
        ax.plot([joint[e[0]][0], joint[e[1]][0]],
                [joint[e[0]][1], joint[e[1]][1]],
                [joint[e[0]][2], joint[e[1]][2]],
                color=(pose_color_list[i][0] / 255, pose_color_list[i][1] / 255, pose_color_list[i][2] / 255))


def swap(joint):
    tmp = np.zeros_like(joint)
    tmp[:, :, :, 0] = joint[:, :, :, 0]
    tmp[:, :, :, 1] = joint[:, :, :, 2]
    tmp[:, :, :, 2] = joint[:, :, :, 1]
    return tmp


def vis(root_path, output_path):
    pkl_files = [file for file in sorted(os.listdir(root_path)) if file.endswith(".npy")]

    for i in tqdm(range(0, len(pkl_files), 1)):
        motion_seq = []
        people_num = 1
        for j in range(people_num):
            pkl = pkl_files[i + j]
            path = os.path.join(root_path, pkl)
            # Robust load
            joint3d = load_pred_position(path)   # returns (T,24,3)
            joint3d = joint3d.reshape(-1, 24, 3)
            joint3d = np.expand_dims(joint3d, axis=0)
            motion_seq.append(joint3d)

        all_joints3d = np.concatenate(motion_seq, axis=0)
        print(all_joints3d.shape)
        img0_path = os.path.join(img_path, os.path.splitext(pkl_files[i])[0])
        os.makedirs(img0_path, exist_ok=True)

        fig = plt.figure()
        ax = plt.axes(projection='3d')
        ax.xaxis.pane.fill = True
        ax.yaxis.pane.fill = True
        ax.zaxis.pane.fill = True

        ax.set_xlabel('X axis')
        ax.set_ylabel('Y axis')
        ax.set_zlabel('Z axis')
        timestep = list(range(all_joints3d.shape[1]))

        all_joints3d = swap(all_joints3d)

        t1, t2 = 0, 300
        t2_eff = min(t2, all_joints3d.shape[1])
        min_lin = np.min(all_joints3d[:, t1:t2_eff, :, :].reshape(-1, 3), axis=0)
        max_lin = np.max(all_joints3d[:, t1:t2_eff, :, :].reshape(-1, 3), axis=0)

        for k in timestep:
            joints3d = all_joints3d[:, k]
            for joint in joints3d:
                joint = adjust_pose(joint[:, :3])
                ax.scatter(joint[:, 0], joint[:, 1], joint[:, 2], color='black', s=10)
                plot_line(joint, ax)

            ax.set_xlim(min_lin[0], max_lin[0])
            ax.set_ylim(min_lin[1], max_lin[1])
            ax.set_zlim(min_lin[2], max_lin[2])

            ax.set_xticklabels([])
            ax.set_yticklabels([])
            ax.set_zticklabels([])

            ax.set_xlabel('X axis')
            ax.set_ylabel('Y axis')
            ax.set_zlabel('Z axis')

            ax.view_init(elev=0, azim=-90)
            plt.savefig(os.path.join(img0_path, f'{k}.png'))
            plt.cla()

        video0_path = os.path.join(output_path, f'{os.path.splitext(pkl_files[i])[0]}_0.mp4')
        cmd = f"ffmpeg -r 30 -i {img0_path}/%d.png -vb 20M -vcodec mpeg4 -y {video0_path} -loglevel quiet"
        os.system(cmd)


if __name__ == '__main__':
    vis(pkl_path, video_path)


In [ ]:
import os
import re
import shlex
import shutil
import subprocess

img_path = '/content/drive/MyDrive/HCI Project/Datasets/ibdts_processed_data/ibdts_processed_data/3D_Motion_Images'
video_path = '/content/drive/MyDrive/HCI Project/Datasets/ibdts_processed_data/ibdts_processed_data/3D_Motion_Videos'

os.makedirs(video_path, exist_ok=True)

print("Checking ffmpeg availability...")
ffmpeg_path = shutil.which('ffmpeg')
print("ffmpeg found at:", ffmpeg_path)
if ffmpeg_path is None:
    print("ffmpeg does not appear to be installed. Install with: !apt update && !apt install -y ffmpeg")
print()

print("Image root path:", img_path)
print("Exists?", os.path.exists(img_path))
if not os.path.exists(img_path):
    raise SystemExit("img_path does not exist - check the path.")

folders = sorted([f for f in os.listdir(img_path) if os.path.isdir(os.path.join(img_path, f))])
print("Number of subfolders found:", len(folders))
print("First 10 folders (if present):", folders[:10])
print()

def detect_pattern(folder_path):
    files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith('.png')])
    if not files:
        return None, files
    sample = files[:10]
    for fn in files:
        m = re.match(r'^(\d+)\.png$', fn)
        if m:
            numeric = m.group(1)
            pad = len(numeric)
            pattern = f"%0{pad}d.png" if pad>1 else "%d.png"
            return pattern, files

    fn0 = files[0]
    m2 = re.match(r'^0+(\d*)\.png$', fn0)
    if m2:
        pad = len(re.match(r'^0+', fn0).group(0))
        pattern = f"%0{pad}d.png"
        return pattern, files
    return "%d.png", files

for folder in folders:
    folder_path = os.path.join(img_path, folder)
    output_video = os.path.join(video_path, f"{folder}.mp4")
    print("Processing folder:", folder_path)

    pattern, files = detect_pattern(folder_path)
    if pattern is None:
        print("  No png files found in folder, skipping.")
        continue

    print("  Sample files (first 20):", files[:20])
    print("  Using ffmpeg pattern:", pattern)

    folder_path_quoted = shlex.quote(folder_path)
    output_video_quoted = shlex.quote(output_video)
    cmd = f"ffmpeg -r 30 -i {folder_path_quoted}/{pattern} -vcodec libx264 -pix_fmt yuv420p -y {output_video_quoted}"
    print("  Running:", cmd)

    # Run and capture output
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print("  Return code:", proc.returncode)
    if proc.stdout:
        print("  ffmpeg stdout:", proc.stdout[:1000])  # print prefix
    if proc.stderr:
        print("  ffmpeg stderr (first 1000 chars):")
        print(proc.stderr[:1000])

    # check if video created
    if os.path.exists(output_video):
        size_mb = os.path.getsize(output_video) / (1024*1024)
        print(f"  Saved video: {output_video}  ({size_mb:.2f} MB)")
    else:
        print(f"  Video not created at {output_video}. Check ffmpeg errors above.")

    print("-" * 60)